In [ ]:
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from sklearn.feature_selection import mutual_info_regression


# run utils.db_setup to set up duckdb database
conn = duckdb.connect('hm.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

# graphs too large for interactivity
pio.renderers.default = 'svg'

## Schema
![](hm_schema.png)

## Tasks

### `rel-hm-user-churn`

Task is to predict 1-week churn for customers.

Binary Classification

Metric: ROCAUC

In [ ]:
%%sql train_df <<
from user_churn_train

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.customer_id.nunique():,} users.')
print()
print('Label Distribution:')
print(train_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = train_df.timestamp.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby(months).customer_id.nunique() / train_df.customer_id.nunique()
)

In [ ]:
%%sql val_df <<
from user_churn_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.customer_id.nunique():,} users.')
print()
print('Label Distribution:')
print(val_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = val_df.timestamp.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print(f'Customers in val seen in train: {len(set(val_df.customer_id) & set(train_df.customer_id)):,}')
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby(months).customer_id.nunique() / val_df.customer_id.nunique()
)

### `rel-hm-item-sales`

Task is to predict the total sales for an article (the sum of prices of the associated transactions) in the next week.

Regression

Metric: MAE

In [ ]:
%%sql train_df <<
from item_sales_train

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.article_id.nunique():,} products.')
print()
print('Temporal dist of samples:')
months = train_df.timestamp.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of articles (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='article_id', keep='last')
    .groupby(months).article_id.nunique() / train_df.article_id.nunique()
)
print()
print('Label Distribution:')
mean = train_df.sales.mean()
std = train_df.sales.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(train_df.sales == 0).mean():.1%}')
print(f'Quantiles: \n{train_df.sales.quantile(np.arange(0, 1.1, 0.1))}')

In [ ]:
%%sql val_df <<
from item_sales_val

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.article_id.nunique():,} products.')
print()
print('Temporal dist of samples:')
months = val_df.timestamp.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of articles (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='article_id', keep='last')
    .groupby(months).article_id.nunique() / val_df.article_id.nunique()
)
print()
print(f'Articles in val seen in train: {len(set(val_df.article_id) & set(train_df.article_id)):,}')
print()
print('Label Distribution:')
mean = val_df.sales.mean()
std = val_df.sales.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(val_df.sales == 0).mean():.1%}')
print(f'Quantiles: \n{val_df.sales.quantile(np.arange(0, 1.1, 0.1))}')

## Article

In [ ]:
%%sql article <<
from article

In [ ]:
article.head()

### Categoricals


In [ ]:
def analyze_cat(df, col):
    print(f'{col.upper()}\n')
    print(f'Unique values: {df[col].nunique()}')
    print()
    print(f'Value counts:\n{df[col].value_counts(normalize=True, dropna=False)}')
    print()

In [ ]:
analyze_cat(article, 'section_name')

In [ ]:
analyze_cat(article, 'garment_group_name')

In [ ]:
analyze_cat(article, 'product_group_name')

In [ ]:
analyze_cat(article, 'department_name')

In [ ]:
analyze_cat(article, 'index_group_name')

In [ ]:
analyze_cat(article, 'index_name')

In [ ]:
analyze_cat(article, 'perceived_colour_master_name')

In [ ]:
analyze_cat(article, 'perceived_colour_value_name')

In [ ]:
analyze_cat(article, 'colour_group_name')

In [ ]:
analyze_cat(article, 'graphical_appearance_name')

In [ ]:
analyze_cat(article, 'product_type_name')

In [ ]:
analyze_cat(article, 'product_code')

In [ ]:
joined = val_df.join(article, on='article_id', rsuffix='_')
mi = mutual_info_regression(
    joined[
        ['product_type_no', 'perceived_colour_master_id', 'graphical_appearance_no',
         'index_group_no', 'department_no', 'section_no', 'garment_group_no', 'colour_group_code']
    ].values,
    joined['sales'].values
)

In [ ]:
pd.Series({
    col: mi 
    for col, mi in zip(
        ['product_type_no', 'perceived_colour_master_id', 'graphical_appearance_no',
         'index_group_no', 'department_no', 'section_no', 'garment_group_no', 'colour_group_code'],
        mi
    )
}).sort_values(ascending=False)

Notes:
- article table pretty much categoricals only
- Many columns overlap
- Probably going for `department_no`, `section_no`, `perceived_colour_master_id` as the others are coarser categorizations of
the same things.

In [ ]:
article.detail_desc.head()

In [ ]:
article.detail_desc.str.len().describe()

## Customer

In [ ]:
%%sql customer <<
from customer

In [ ]:
customer.head()

In [ ]:
customer.FN.value_counts(normalize=True, dropna=False)

In [ ]:
customer.Active.value_counts(normalize=True, dropna=False)

In [ ]:
customer.club_member_status.value_counts(normalize=True, dropna=False)

In [ ]:
customer.fashion_news_frequency.value_counts(normalize=True, dropna=False)

In [ ]:
px.histogram(customer, x='age', nbins=60, title='Age Distribution', histnorm='percent')

In [ ]:
customer.postal_code.nunique()

## Transactions

In [ ]:
%%sql transactions <<
from transactions

In [ ]:
transactions.head()

In [ ]:
print(f'{len(transactions):,} transactions')

In [ ]:
by_article = transactions.groupby('article_id').price.sum()

In [ ]:
px.histogram(
    x=(by_article + 1), title='Sales per Article', histnorm='percent', log_y=True,
    log_x=True, labels={'x': 'Sales'}
)

In [ ]:
cum_sales = by_article.sort_values().cumsum() / by_article.sum()
px.line(
    x=np.arange(len(cum_sales)) / len(cum_sales), y=cum_sales, title='Lorenz Curve (articles)',
    labels={'x': 'Fraction of articles', 'y': 'Fraction of sales'}
)

^ Pareto principle

In [ ]:
by_customer = transactions.groupby('customer_id').price.sum()

In [ ]:
px.histogram(
    x=(by_customer + 1), title='Purchase Amount per Customer', histnorm='percent', log_y=True,
    log_x=True, labels={'x': 'Purchase Amount'}
)

In [ ]:
cum_sales = by_customer.sort_values().cumsum() / by_article.sum()
px.line(
    x=np.arange(len(cum_sales)) / len(cum_sales), y=cum_sales, title='Lorenz Curve (customers)',
    labels={'x': 'Fraction of customers', 'y': 'Fraction of purchase amount'}
)

In [ ]:
transactions.sales_channel_id.value_counts(normalize=True, dropna=False)

In [ ]:
transactions.groupby('sales_channel_id').price.agg(['mean', 'size', 'sum'])

In [ ]:
%%sql txns_by_month <<
select
    strftime('%Y-%m', t_dat) as month,
    count(*) as num_sales,
    sum(price) as sold_amount
from transactions
group by month

In [ ]:
px.bar(txns_by_month, x='month', y='num_sales', title='Number of Sales by Month')

In [ ]:
px.bar(txns_by_month, x='month', y='sold_amount', title='Sold Amount by Month')

In [ ]:
%%sql variability <<
with ids as (
    select article_id from article using sample 100
)
select
    ids.article_id,
    date_part('week', t.t_dat) as week,
    count(*) as num_sales,
    sum(t.price) as sold_amount,
    avg(t.price) as avg_price
from ids
left join transactions as t
    on ids.article_id = t.article_id
where
    t.t_dat between '2019-01-01'::timestamp and '2019-12-31'::timestamp
group by ids.article_id, week
order by ids.article_id, week asc

In [ ]:
first_weeks = variability.groupby('article_id').week.first()

In [ ]:
valid = first_weeks[first_weeks == 1].index

In [ ]:
valid

In [ ]:
variability = variability[variability.article_id.isin(valid)]
first_weeks = first_weeks[first_weeks == 1]
variability = variability.set_index(['article_id', 'week'])
first_week_values = variability.loc[list(zip(first_weeks.index, first_weeks))]

In [ ]:
first_week_values.droplevel('week', 0)

In [ ]:
normed_var = np.log2(variability / first_week_values.droplevel('week', 0))

In [ ]:
px.box(
    normed_var.reset_index(), x='week', y='sold_amount',
    title='Log normalized sold amount per week',
)

In [ ]:
px.box(
    normed_var.reset_index(), x='week', y='num_sales',
    title='Log normalized num sales per week',
)

In [ ]:
px.box(
    normed_var.reset_index(), x='week', y='avg_price',
    title='Log normalized average price per week',
)

Notes:
- Seasonality: sales peak in summer months
- Very high variability in number/amount of sales week-over-week
- Prices are more stable but do exhibit some variability downwards (ie discounts)

In [ ]:
conn.close()